# LLM Agentic Legal Information Retrieval — Offline Reproducible Notebook**Pipeline:** BM25 + Dense (multilingual-e5-large) + Cross-Encoder rerank (mmarco-mMiniLMv2) + drop layer (operative-vs-dispositif regex) + boilerplate-ADD (val-grounded).**No external API calls.** All models loaded from Kaggle datasets. Reproducible offline within 12-hour Kaggle notebook limit.**Expected output:** `/kaggle/working/submission.csv`

In [ ]:
# -*- coding: utf-8 -*-
"""
CELL 1 — Setup, imports, and data loading.

Offline Kaggle notebook for LLM Agentic Legal Information Retrieval competition.
All assets loaded from Kaggle datasets — no external API calls.

Required Kaggle inputs (mount as datasets):
  /kaggle/input/llm-agentic-legal-information-retrieval/
    ├── train.csv, val.csv, test.csv
    ├── laws_de.csv
    └── court_considerations.csv
  /kaggle/input/multilingual-e5-large/   (intfloat/multilingual-e5-large, 560MB)
  /kaggle/input/mmarco-mminilmv2/        (cross-encoder/mmarco-mMiniLMv2-L12-H384-v1, 120MB)

Output:
  /kaggle/working/submission.csv
"""
import os, sys, re, gc, time, json
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

# Runtime detection
IS_KAGGLE = os.path.exists('/kaggle')
if IS_KAGGLE:
    DATA = Path('/kaggle/input/llm-agentic-legal-information-retrieval')
    E5_DIR = Path('/kaggle/input/multilingual-e5-large')
    RERANK_DIR = Path('/kaggle/input/mmarco-mminilmv2')
    OUT_PATH = Path('/kaggle/working/submission.csv')
else:
    DATA = Path('Data')
    E5_DIR = Path('models/multilingual-e5-large')
    RERANK_DIR = Path('models/mmarco-mMiniLMv2-L12-H384-v1')
    OUT_PATH = Path('submission.csv')

print(f'IS_KAGGLE: {IS_KAGGLE}')
print(f'DATA: {DATA}')
print(f'OUT: {OUT_PATH}')

# Load competition data
train = pd.read_csv(DATA / 'train.csv')
val = pd.read_csv(DATA / 'val.csv')
test = pd.read_csv(DATA / 'test.csv')
laws = pd.read_csv(DATA / 'laws_de.csv').fillna('')

print(f'\nLoaded:')
print(f'  train: {len(train)} queries (mean {train["gold_citations"].astype(str).str.split(";").str.len().mean():.1f} cits/q)')
print(f'  val:   {len(val)} queries (mean {val["gold_citations"].astype(str).str.split(";").str.len().mean():.1f} cits/q)')
print(f'  test:  {len(test)} queries')
print(f'  laws:  {len(laws):,} entries')

# Notebook self-translates English queries to German via Helsinki-NLP/opus-mt-en-de
# OR uses pre-translated val_queries_de.csv / test_queries_de.csv if available
# For reproducibility we include the translation step in the pipeline


## Translation: English → GermanQueries arrive in English. The corpus (laws_de.csv and court_considerations.csv) is German+French. We translate queries to German for better lexical match with the corpus.

In [ ]:
# -*- coding: utf-8 -*-
"""
CELL 2 — Translation: English query → German.

Uses Helsinki-NLP/opus-mt-en-de loaded from Kaggle dataset.
Queries are translated once and cached.
"""
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'DEVICE: {DEVICE}')

if IS_KAGGLE:
    TRANS_DIR = Path('/kaggle/input/opus-mt-en-de')
else:
    TRANS_DIR = Path('models/opus-mt-en-de')

print(f'Loading translation model from {TRANS_DIR}...')
t0 = time.time()
tok_t = AutoTokenizer.from_pretrained(str(TRANS_DIR))
mdl_t = AutoModelForSeq2SeqLM.from_pretrained(str(TRANS_DIR)).to(DEVICE).eval()
print(f'  loaded in {time.time()-t0:.0f}s')

def translate_batch(texts, batch_size=4, max_length=512):
    """Translate English texts to German in batches."""
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        # Truncate long inputs (queries can be 1500+ chars)
        enc = tok_t(batch, return_tensors='pt', padding=True, truncation=True, max_length=max_length).to(DEVICE)
        with torch.no_grad():
            gen = mdl_t.generate(**enc, max_length=max_length, num_beams=2)
        out.extend(tok_t.batch_decode(gen, skip_special_tokens=True))
    return out

# Translate test queries
t0 = time.time()
test_de = translate_batch(test['query'].tolist(), batch_size=2)
test['query_de'] = test_de
print(f'Translated {len(test_de)} test queries in {time.time()-t0:.0f}s')

# Translate val queries (for FT and for boilerplate-ADD calibration)
val_de = translate_batch(val['query'].tolist(), batch_size=2)
val['query_de'] = val_de
print(f'Translated {len(val_de)} val queries in {time.time()-t0:.0f}s')

# Free translation model VRAM
del mdl_t, tok_t
gc.collect()
torch.cuda.empty_cache()
print('Translation model unloaded.')


## Retrieval Stage 1: BM25 over laws + courtBM25 captures explicit-vocabulary matches. Also runs explicit-mention extraction: queries that mention "Art. N CODE" boost that cit into the candidate pool.

In [ ]:
# -*- coding: utf-8 -*-
"""
CELL 3 — FAST BM25 retrieval over laws + court (sparse, vectorized).

Replaces rank_bm25.BM25Okapi (pure-Python, O(corpus) per query — the bottleneck that
hung on the 1.98M-row court corpus) with a scipy-sparse BM25 that precomputes a
BM25-weighted doc-term matrix and scores ALL queries in ONE sparse matmul. Court
aggregation is vectorized (pandas groupby, not .iterrows over 2.47M rows). Scores
test+val (val needed for the macro-F1 readout in the final cell).
"""
import time, scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer

TOK_PAT = re.compile(r'[\wäöüÄÖÜß.]+', re.UNICODE)
def tok_bm25(s):
    return [t.lower() for t in TOK_PAT.findall(str(s))]   # kept for any downstream use

class FastBM25:
    """Sparse BM25 (k1=1.5, b=0.75). Tokenizes via CountVectorizer (C-level, keeps legal
    notation like 'Art.'), precomputes a BM25-weighted matrix, scores queries by sparse matmul."""
    def __init__(self, texts, k1=1.5, b=0.75, min_df=1):
        self.cv = CountVectorizer(lowercase=True, token_pattern=r'(?u)[\wäöüÄÖÜß.]+', min_df=min_df)
        X = self.cv.fit_transform(texts).astype(np.float32).tocsr(); N = X.shape[0]
        df = np.asarray((X > 0).sum(0)).ravel()
        idf = np.log(1.0 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
        dl = np.asarray(X.sum(1)).ravel().astype(np.float32); avgdl = float(dl.mean()) or 1.0
        Bd = (1.0 - b + b * dl / avgdl).astype(np.float32)
        coo = X.tocoo(); tf = coo.data
        w = idf[coo.col] * (tf * (k1 + 1.0)) / (tf + k1 * Bd[coo.row])
        self.W = sp.csr_matrix((w.astype(np.float32), (coo.row, coo.col)), shape=X.shape)
    def topk(self, queries, K):
        Q = self.cv.transform(queries).astype(np.float32); Q.data[:] = 1.0   # binary query terms
        S = (self.W @ Q.T).toarray()                                          # (N_docs, n_queries)
        out = []
        for j in range(S.shape[1]):
            col = S[:, j]; k = min(K, len(col)); idx = np.argpartition(-col, k - 1)[:k]
            idx = idx[np.argsort(-col[idx])]
            out.append([(int(i), float(col[i])) for i in idx])
        return out

# queries: test + val (val drives the macro-F1 readout in the final cell)
Q_ROWS = list(test.iterrows()) + list(val.iterrows())
Q_IDS  = [r['query_id'] for _, r in Q_ROWS]
Q_DE   = [str(r['query_de']) for _, r in Q_ROWS]

# --- laws BM25 ---
t0 = time.time()
laws_text = (laws['citation'].astype(str) + ' ' + laws['text'].astype(str) + ' ' + laws['title'].astype(str)).tolist()
laws_cits = laws['citation'].astype(str).tolist()
laws_set  = set(laws_cits)
bm25_laws = FastBM25(laws_text)
BM25_LAWS_K = 200
law_top = bm25_laws.topk(Q_DE, BM25_LAWS_K)
bm25_laws_per_q = {Q_IDS[j]: [(laws_cits[i], s) for i, s in law_top[j]] for j in range(len(Q_IDS))}
print(f'laws BM25 (fast, {len(laws_cits):,} docs) + scored {len(Q_IDS)} queries in {time.time()-t0:.0f}s')

# --- court BM25 (aggregate by citation via vectorized groupby) ---
t0 = time.time()
cdf = pd.concat([c[['citation', 'text']] for c in pd.read_csv(DATA / 'court_considerations.csv', chunksize=500_000)],
                ignore_index=True)
cdf['text'] = cdf['text'].fillna('').astype(str)
g = cdf.groupby('citation', sort=False)['text'].agg(' '.join)
court_cits = g.index.tolist(); court_text = g.tolist()
court_by_cit = {c: [t] for c, t in zip(court_cits, court_text)}   # cell-5 reranker compat (' '.join([t]) == t)
del cdf, g; gc.collect()
print(f'  court aggregated {len(court_cits):,} refs in {time.time()-t0:.0f}s')
t0 = time.time()
bm25_court = FastBM25(court_text, min_df=2)   # min_df=2 prunes hapax on the huge court vocab (negligible BM25 effect, big mem/speed win)
del court_text; gc.collect()
BM25_COURT_K = 300
court_top = bm25_court.topk(Q_DE, BM25_COURT_K)
bm25_court_per_q = {Q_IDS[j]: [(court_cits[i], s) for i, s in court_top[j]] for j in range(len(Q_IDS))}
print(f'  court BM25 (fast) built + scored {len(Q_IDS)} queries in {time.time()-t0:.0f}s')

# --- explicit-mention extraction (test+val) ---
MENTION_PAT = re.compile(r'Art\.\s*(\d+[a-z]?)(?:\s+Abs\.\s*(\d+[a-z]*))?(?:\s+lit\.\s*[a-z]+)?\s+([A-ZÄÖÜ][A-Za-zÄÖÜäöü]+\.?)', re.UNICODE)
def extract_mentions(qtext):
    out = []
    for m in MENTION_PAT.finditer(qtext):
        a, ab, co = m.groups()
        if ab: out.append(f'Art. {a} Abs. {ab} {co}')
        out.append(f'Art. {a} {co}')
    return out
mention_per_q = {}
for _, row in Q_ROWS:
    ms = set(extract_mentions(str(row['query'])) + extract_mentions(str(row['query_de'])))
    mention_per_q[row['query_id']] = [c for c in ms if c in laws_set]
print(f'mentions: {sum(len(v) for v in mention_per_q.values())} corpus-valid across {len(Q_IDS)} queries')


## Retrieval Stage 2: Dense (multilingual-e5-large)Dense semantic retrieval over laws_de. Captures cross-lingual semantic matches that BM25 misses. Court corpus too large for dense (2.47M docs) — sticks with BM25 there.

In [ ]:
# -*- coding: utf-8 -*-
"""
CELL 4 — Dense retrieval over laws_de using multilingual-e5-large.

For each test query: encode query → top-200 laws by cosine similarity.
Court refs are too many (2.47M) for dense — stick with BM25 for those.
"""
import torch
from transformers import AutoTokenizer, AutoModel

print(f'Loading e5 model from {E5_DIR}...')
t0 = time.time()
tok_e = AutoTokenizer.from_pretrained(str(E5_DIR))
mdl_e = AutoModel.from_pretrained(str(E5_DIR)).to(DEVICE).eval()
print(f'  loaded in {time.time()-t0:.0f}s, params {sum(p.numel() for p in mdl_e.parameters())/1e6:.0f}M')

def mean_pool(h, mask):
    mask = mask.unsqueeze(-1).float()
    return (h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

def encode(texts, prefix='passage: ', batch_size=8, max_length=512):
    """Encode texts to L2-normalized embeddings."""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = [prefix + str(t) for t in texts[i:i+batch_size]]
        enc = tok_e(batch, return_tensors='pt', padding=True, truncation=True, max_length=max_length).to(DEVICE)
        with torch.no_grad():
            out = mdl_e(**enc)
        embs = mean_pool(out.last_hidden_state, enc['attention_mask'])
        embs = torch.nn.functional.normalize(embs, p=2, dim=1)
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs).astype('float32')

# Encode laws_de
print('\nEncoding laws_de...')
t0 = time.time()
laws_embs = encode(laws_text, prefix='passage: ', batch_size=16)
print(f'  laws encoded: {laws_embs.shape} in {time.time()-t0:.0f}s')

# Encode test queries (German)
print('\nEncoding test queries...')
test_q_embs = encode(test['query_de'].tolist(), prefix='query: ', batch_size=4)

# Encode val queries (for later use)
val_q_embs = encode(val['query_de'].tolist(), prefix='query: ', batch_size=4)

# Free e5 model
del mdl_e, tok_e
gc.collect()
torch.cuda.empty_cache()

# Cosine sim search (dot product since both are L2-normalized)
DENSE_LAWS_K = 200

allq_embs = np.vstack([test_q_embs, val_q_embs])
allq_ids  = test['query_id'].tolist() + val['query_id'].tolist()
dense_laws_per_q = {}
for i, qid in enumerate(allq_ids):
    sims = laws_embs @ allq_embs[i]
    top_idx = np.argsort(-sims)[:DENSE_LAWS_K]
    dense_laws_per_q[qid] = [(laws_cits[j], float(sims[j])) for j in top_idx]
print(f'\nDense retrieval done for {len(allq_ids)} queries (test+val)')

# Free laws_embs (large)
del laws_embs
gc.collect()


## Fusion + Cross-Encoder RerankingReciprocal Rank Fusion (RRF) merges BM25 laws + BM25 court + dense laws + explicit mentions. Top-150 candidates are reranked by mmarco-mMiniLMv2 cross-encoder.

In [ ]:
# -*- coding: utf-8 -*-
"""
CELL 5 — Fusion + cross-encoder reranking.

For each test query:
  1. Reciprocal Rank Fusion (RRF) of BM25 laws + BM25 court + Dense laws + mentions
  2. Cross-encoder rerank of top-150 candidates
  3. Pick top-K (calibrated on val)
"""
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

RRF_K_CONST = 60
RERANK_K = 150

def rrf_fuse(*rankings, k=RRF_K_CONST):
    """RRF fusion of multiple (cit, score) ranked lists.
    Returns dict {cit: rrf_score}.
    """
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, (cit, _) in enumerate(ranking):
            scores[cit] += 1.0 / (k + rank + 1)
    return scores

# --- Build laws_text lookup for reranker ---
laws_text_map = {laws['citation'].iloc[i]: laws_text[i] for i in range(len(laws))}
# Note: court_by_cit already maps citation → list of paragraph texts
def court_text_concat(cit):
    rows = court_by_cit.get(cit, [])
    return ' '.join(rows)[:1500]

def fuse_per_q(qid):
    rankings = [
        bm25_laws_per_q.get(qid, []),
        bm25_court_per_q.get(qid, []),
        dense_laws_per_q.get(qid, []),
        # Mentions get top-rank boost
        [(c, 1.0) for c in mention_per_q.get(qid, [])],
    ]
    return rrf_fuse(*rankings)

# --- Load cross-encoder ---
print(f'Loading cross-encoder from {RERANK_DIR}...')
t0 = time.time()
tok_r = AutoTokenizer.from_pretrained(str(RERANK_DIR))
mdl_r = AutoModelForSequenceClassification.from_pretrained(str(RERANK_DIR)).to(DEVICE).eval()
print(f'  loaded in {time.time()-t0:.0f}s')

def rerank(query, candidates, top_n=RERANK_K, batch_size=32, max_length=384):
    """Rerank (cit, text) pairs using cross-encoder."""
    if not candidates:
        return []
    pairs = [(query, text) for _, text in candidates]
    scores = []
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i+batch_size]
        enc = tok_r([q for q, _ in batch], [t for _, t in batch],
                    return_tensors='pt', padding=True, truncation=True, max_length=max_length).to(DEVICE)
        with torch.no_grad():
            out = mdl_r(**enc)
        s = out.logits.squeeze(-1).cpu().numpy()
        scores.extend(s.tolist())
    scored = [(candidates[j][0], scores[j]) for j in range(len(candidates))]
    scored.sort(key=lambda x: -x[1])
    return scored[:top_n]

# --- Per-query: fuse, get text for top-RERANK_K, rerank ---
court_pat = re.compile(r'^(BGE|\d+[A-Z]_)')

def get_cit_text(cit):
    if court_pat.match(cit):
        return court_text_concat(cit)
    return laws_text_map.get(cit, '')

ranked_per_q = {}
_ALLQ = list(test.iterrows()) + list(val.iterrows())
print(f'\nFusing + reranking {len(_ALLQ)} queries (test+val)...')
t0 = time.time()
for _, row in _ALLQ:
    qid = row['query_id']
    qde = str(row['query_de'])
    fused = fuse_per_q(qid)
    # Top RERANK_K candidates by fusion score
    top_cands = sorted(fused.items(), key=lambda x: -x[1])[:RERANK_K]
    cand_with_text = [(c, get_cit_text(c)) for c, _ in top_cands]
    reranked = rerank(qde, cand_with_text, top_n=RERANK_K)
    ranked_per_q[qid] = reranked
print(f'  reranked in {time.time()-t0:.0f}s')

# Free reranker
del mdl_r, tok_r
gc.collect()
torch.cuda.empty_cache()


## Drop Layer (operative-vs-dispositif content classification)Drops court refs whose paragraph text is non-doctrinal: issue-framing transitions, party-argument summaries, case-specific procedural admissibility, pure fact narratives. All rules grounded in val analysis. Pure regex over court_considerations.csv text.

In [ ]:
# -*- coding: utf-8 -*-
"""
CELL 6 — Drop layer: operative-vs-dispositif content classification.

Replicates v305/v307/v311/v312 mechanism. Drop court refs whose paragraph text
is a NON-DOCTRINAL content type:
  1. Issue-framing transition ("Streitig und zu prüfen ist", "Zu prüfen bleibt")
  2. Party-argument summary ("Beschwerdeführer macht geltend", "rügt", "se plaint")
  3. Pure fact narrative (dates + amounts only, no doctrinal vocabulary)
  4. Case-specific procedural admissibility (Eventualbegründung, "nicht einzugehen")
  5. Gehör digression with different sub-grievances

All rules grounded in val analysis. No LLM API needed — pure regex over corpus text.
"""

# --- Drop pattern regexes (operative-vs-dispositif content class) ---
DROP_PAT_ISSUE_FRAMING = re.compile(
    r'(Streitig\s+(und\s+zu\s+prüfen\s+ist|ist)|'
    r'Zu\s+prüfen\s+bleibt|'
    r'Es\s+bleibt\s+zu\s+untersuchen|'
    r'vorab\s+die\s+von\s+Amtes\s+wegen\s+zu\s+prüfende\s+Rechtsfrage|'
    r'^[0-9]+\.[0-9]?\.?\s*Streitig)',
    re.IGNORECASE
)

DROP_PAT_PARTY_SUMMARY = re.compile(
    r'(Der\s+Beschwerdeführer\s+(macht\s+geltend|rügt|bemängelt|kritisiert|beanstandet)|'
    r'Sie\s+macht\s+geltend|'
    r'Die\s+Beschwerdeführerin\s+(macht\s+geltend|rügt|bemängelt)|'
    r'Le\s+recourant\s+(se\s+plaint|critique|invoque|fait\s+valoir)|'
    r'La\s+recourante\s+(critique|se\s+plaint|fait\s+valoir)|'
    r'Zur\s+Hauptsache\s+erblickt\s+der\s+Kläger)',
    re.IGNORECASE
)

DROP_PAT_CASE_PROC = re.compile(
    r'(in\s+der\s+Eventualbegründung|'
    r'mangels\s+Tatsachenbehauptungen|'
    r'ist\s+nicht\s+einzugehen|'
    r'liess\s+die\s+Vorinstanz\s+offen|'
    r'Die\s+Kritik\s+der\s+Vorinstanz\s+am\s+rückweisenden\s+Urteil)',
    re.IGNORECASE
)

# Pure fact narrative: short text with dates+amounts but no doctrinal "Art." references
DATE_AMT_PAT = re.compile(r'(\d{1,2}\.\s*[A-Z][a-zü]+\s+\d{4}|CHF\s*[\d\']+|€\s*[\d\']+|\d+[%‰])', re.IGNORECASE)
ART_PAT_IN_TEXT = re.compile(r'\bArt\.\s*\d+', re.IGNORECASE)

def is_dispositif_class(text):
    """Return classification name if text matches a drop class, else None."""
    if not text or len(text) > 700:  # Long substantive paragraphs are safe
        return None
    if DROP_PAT_ISSUE_FRAMING.search(text):
        return 'issue_framing'
    if DROP_PAT_PARTY_SUMMARY.search(text):
        return 'party_summary'
    if DROP_PAT_CASE_PROC.search(text):
        return 'case_procedural'
    # Pure fact narrative: many dates/amounts, no Art. references
    n_dates = len(DATE_AMT_PAT.findall(text))
    n_arts = len(ART_PAT_IN_TEXT.findall(text))
    if n_dates >= 3 and n_arts == 0 and len(text) < 500:
        return 'fact_narrative'
    return None

# --- Apply drops to reranked per_q ---
n_drops_total = 0
drop_breakdown = Counter()
ranked_per_q_dropped = {}

for qid, ranked in ranked_per_q.items():
    new_ranked = []
    for cit, score in ranked:
        if court_pat.match(cit):
            text = court_text_concat(cit)
            cls = is_dispositif_class(text)
            if cls is not None:
                drop_breakdown[cls] += 1
                n_drops_total += 1
                continue
        new_ranked.append((cit, score))
    ranked_per_q_dropped[qid] = new_ranked

print(f'\nDrop layer applied:')
print(f'  total drops: {n_drops_total}')
for cls, n in drop_breakdown.most_common():
    print(f'    {cls:<20} {n}')

ranked_per_q = ranked_per_q_dropped


## Boilerplate-ADD Layer (val-grounded breakthrough)**The mechanism:** val gold shows LEXam annotators systematically add procedural-boilerplate articles based on query case type. Train has 4.1 gold cits/q; val has 25.1 cits/q. 9/10 val queries cite Art. 100 Abs. 1 BGG (Supreme Court 30-day appeal deadline).**Confirmed on public LB:** +0.024 lift from this layer alone (v313/v314).

In [ ]:
# -*- coding: utf-8 -*-
"""
CELL 7 — Boilerplate-ADD layer + VAL-calibrated K + val macro-F1 readout + submission.

Universal: Art. 100 Abs. 1 BGG (9/10 val). Case-type cluster cit added by query type.
K is calibrated on val (not hardcoded), and we PRINT val macro-F1 so the offline score is
confirmed WITHOUT a Kaggle submission. (Note: v315 showed broad cluster ADD can regress on
LB — clusters are kept here as the v314 mechanism; flip ADD_CLUSTERS=False to test universal-only.)
"""
ADD_CLUSTERS = True
def classify_case_type(q):
    qlow = q.lower(); types = set()
    if any(k in q for k in ['StPO','Strafverfahren','StGB','Untersuchungshaft','Beschuldigt','Anklage','Verbrechen','Strafgesetzbuch']) or \
       any(k in qlow for k in ['criminal','crime','theft','assault','detention','pretrial','pre-trial','accused','cpp ','dna profile','sexual','prosecution','penal']):
        types.add('criminal')
    if any(k in q for k in ['ATSG','IVG','UVG','AHVG','BVG','KVG','IV-Stelle','Versicherung','Invaliden','AHV']) or \
       any(k in qlow for k in ['social insurance','pension','invalidity','accident','occupational','disabili','rehab','lai ','laa ','lava']):
        types.add('social_insurance')
    if any(k in q for k in ['IPRG','Internationale','Lugano','EuGVVO']) or any(k in qlow for k in ['international','foreign','cross-border']):
        types.add('international')
    if any(k in q for k in ['Scheidung','Ehegatten','Unterhalt','Kindes','Sorgerecht','Vorsorgeunterhalt']) or \
       any(k in qlow for k in ['marriage','divorce','spousal','maintenance','custody','child support','parental']):
        types.add('civil_family')
    return types or {'other'}
PRIORITY = ['criminal','social_insurance','international','civil_family','other']
CLUSTER_CIT = {'criminal':'Art. 428 Abs. 1 StPO','social_insurance':'Art. 8 Abs. 1 ATSG',
               'international':'Art. 100 Abs. 1 IPRG','civil_family':'Art. 285 Abs. 1 ZGB','other':'Art. 29 Abs. 2 BV'}
UNIVERSAL_CIT = 'Art. 100 Abs. 1 BGG'

def build_preds(rows, K):
    preds = {}
    for _, row in rows:
        qid = row['query_id']; ranked = ranked_per_q.get(qid, [])
        picks = [c for c, _ in ranked[:K]]; ps = set(picks)
        if UNIVERSAL_CIT not in ps: picks.append(UNIVERSAL_CIT); ps.add(UNIVERSAL_CIT)
        if ADD_CLUSTERS:
            for ct in PRIORITY:
                if ct in classify_case_type(str(row['query'])):
                    cc = CLUSTER_CIT[ct]
                    if cc not in ps: picks.append(cc); ps.add(cc)
                    break
        preds[qid] = picks
    return preds

def _cits(s): return [x.strip() for x in str(s).split(';') if x.strip()]
def macro_f1(preds, gold):
    fs = []
    for q, g in gold.items():
        p = set(preds.get(q, [])); gg = set(g)
        if not p and not gg: fs.append(1.0); continue
        if not p or not gg: fs.append(0.0); continue
        tp = len(p & gg); pr = tp / len(p); rc = tp / len(gg)
        fs.append(0.0 if pr + rc == 0 else 2 * pr * rc / (pr + rc))
    return float(np.mean(fs))

# --- calibrate K on val + report val macro-F1 (offline score confirmation) ---
val_gold = {r['query_id']: set(_cits(r['gold_citations'])) for _, r in val.iterrows()}
val_rows = list(val.iterrows())
best = (0.0, 25)
for K in range(8, 41):
    f1 = macro_f1(build_preds(val_rows, K), val_gold)
    if f1 > best[0]: best = (f1, K)
BEST_F1, BEST_K = best
print(f'>>> VAL macro-F1 = {BEST_F1:.4f} @ K={BEST_K}  (pure offline pipeline; ADD_CLUSTERS={ADD_CLUSTERS})')
ru = macro_f1({r['query_id']: [c for c,_ in ranked_per_q.get(r['query_id'],[])[:BEST_K]] for _,r in val_rows}, val_gold)
print(f'    val F1 retrieval-only@{BEST_K} = {ru:.4f}  (+boilerplate ADD = {BEST_F1:.4f})')

# --- write test submission at calibrated K ---
test_preds = build_preds(list(test.iterrows()), BEST_K)
order = test['query_id'].tolist()
sub = pd.DataFrame([{'query_id': q, 'predicted_citations': ';'.join(test_preds[q])} for q in order])
sub.to_csv(OUT_PATH, index=False)
print(f'Wrote {OUT_PATH}: {len(sub)} queries, mean {sub["predicted_citations"].str.split(";").str.len().mean():.1f} cits/q')


## DoneSubmission written to `/kaggle/working/submission.csv`. Reproducibility:- No external API calls ✓- All models from Kaggle datasets ✓- 12hr runtime budget: ~15-20 min actual ✓- Deterministic given fixed weights ✓